In [1]:
from dask.distributed import Client, LocalCluster
import dask
import dask.dataframe as dd
import pandas as pd
import time
import logging
import json
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

dask.config.set({
    "distributed.worker.memory.target": 0.60,
    "distributed.worker.memory.spill": 0.70,
    "distributed.worker.memory.pause": 0.80,
    "distributed.worker.memory.terminate": 0.98
})

cluster = LocalCluster(
    n_workers=1,
    threads_per_worker=2,
    processes=False,
    memory_limit="6GB",
    dashboard_address=":8787",
    silence_logs=logging.ERROR 
)

client = Client(cluster)
logger.info(f"Dask dashboard: {client.dashboard_link}")

2026-03-19 01:37:07,467 - INFO - Dask dashboard: http://192.168.0.29:8787/status


In [2]:
dask_total_start = time.time()
silver_start = time.time()

weather = dd.read_parquet(
    "../data/bronze/weather_raw.parquet",
    columns=["Datum", "Tid (UTC)", "Lufttemperatur", "station_id", "price_zone"]
)

energy = dd.read_parquet(
    "../data/bronze/energy_raw.parquet",
    columns=["datetime", "date", "hour", "year", "month", "price_zone", "spot_price_sek", "consumption_mwh"]
)

logger.info(f"Partitions: weather={weather.npartitions}, energy={energy.npartitions}")

weather["Lufttemperatur"] = dd.to_numeric(weather["Lufttemperatur"], errors="coerce")

weather["timestamp"] = dd.to_datetime(
    weather["Datum"].astype(str) + " " + weather["Tid (UTC)"].astype(str),
    errors="coerce"
)

weather_clean = weather[
    weather["Lufttemperatur"].notnull() &
    weather["timestamp"].notnull() &
    (weather["Lufttemperatur"] >= -60) &
    (weather["Lufttemperatur"] <= 45)
]

weather_clean = weather_clean.assign(
    temperature=weather_clean["Lufttemperatur"],
    hour=weather_clean["timestamp"].dt.hour,
    month=weather_clean["timestamp"].dt.month,
    year=weather_clean["timestamp"].dt.year,
    date=weather_clean["timestamp"].dt.strftime("%Y-%m-%d")
)

weather_clean = weather_clean[[
    "station_id",
    "price_zone",
    "timestamp",
    "temperature",
    "hour",
    "month",
    "year",
    "date"
]]

energy = energy.assign(
    date=dd.to_datetime(energy["date"], errors="coerce").dt.strftime("%Y-%m-%d"),
    hour=dd.to_numeric(energy["hour"], errors="coerce"),
    year=dd.to_numeric(energy["year"], errors="coerce"),
    month=dd.to_numeric(energy["month"], errors="coerce"),
    spot_price_sek=dd.to_numeric(energy["spot_price_sek"], errors="coerce"),
    consumption_mwh=dd.to_numeric(energy["consumption_mwh"], errors="coerce")
)

os.makedirs("../data/silver_dask", exist_ok=True)
weather_clean.to_parquet("../data/silver_dask/weather_clean", write_index=False)

silver_time = time.time() - silver_start
logger.info(f"Dask Silver done in {silver_time:.1f}s")

2026-03-19 01:37:12,459 - INFO - Partitions: weather=20, energy=1
2026-03-19 01:40:39,926 - INFO - Dask Silver done in 207.6s


In [4]:
gold_start = time.time()

hourly = weather_clean.groupby(
    ["price_zone", "date", "hour", "year", "month"]
).agg({
    "temperature": ["mean", "min", "max", "count"]
}).compute()

hourly.columns = ["avg_temp", "min_temp", "max_temp", "station_count"]
hourly = hourly.reset_index()

energy_pd = energy.compute()

hourly["price_zone"] = hourly["price_zone"].astype(str)
energy_pd["price_zone"] = energy_pd["price_zone"].astype(str)

hourly["date"] = pd.to_datetime(hourly["date"], errors="coerce").dt.strftime("%Y-%m-%d")
energy_pd["date"] = pd.to_datetime(energy_pd["date"], errors="coerce").dt.strftime("%Y-%m-%d")

hourly["hour"] = pd.to_numeric(hourly["hour"], errors="coerce").astype("Int64")
energy_pd["hour"] = pd.to_numeric(energy_pd["hour"], errors="coerce").astype("Int64")

hourly = hourly.dropna(subset=["date", "hour", "price_zone"])
energy_pd = energy_pd.dropna(subset=["date", "hour", "price_zone"])

gold = hourly.merge(
    energy_pd,
    on=["price_zone", "date", "hour"],
    how="inner"
)

gold = gold.sort_values(["price_zone", "date", "hour"])

gold["rolling_24h_avg_temp"] = (
    gold.groupby("price_zone")["avg_temp"]
    .transform(lambda x: x.rolling(24, min_periods=1).mean())
)

os.makedirs("../data/gold", exist_ok=True)
gold.to_parquet("../data/gold/gold_weather_energy_dask.parquet", index=False)

gold_time = time.time() - gold_start
dask_total_time = time.time() - dask_total_start

logger.info(f"Gold done in {gold_time:.1f}s")
logger.info(f"DASK TOTAL TIME: {dask_total_time:.1f}s")

2026-03-19 02:05:30,819 - INFO - Gold done in 492.6s
2026-03-19 02:05:30,833 - INFO - DASK TOTAL TIME: 1698.5s


In [7]:
benchmark = {
    "spark_total_time_s": None,
    "dask_total_time_s": dask_total_time,
    "spark_lines_of_code": None,
    "dask_lines_of_code": 115,
    "spark_setup_complexity": 'High - JVM, cluster config required',
    "dask_setup_complexity": 'Low - pure Python, pip install',
    "spark_fault_tolerance": 'Built-in - RDD lineage and checkpointing',
    "dask_fault_tolerance": 'Limited - task graph retry only',
    "recommendation": 'Spark preferred at 10GB+ scale'
}

os.makedirs("../docs", exist_ok=True)

with open("../docs/benchmark.json", "w") as f:
    json.dump(benchmark, f, indent=2)
print('Benchmark saved')

Benchmark saved
